Variant Type:
With the introduction of the VARIANT data type, handling semi-structured data has become more streamlined. VARIANT types are designed to store data that doesn’t conform to a fixed schema, such as JSON or XML, directly within a DataFrame column.

Features of VARIANT in PySpark - Flexibility: VARIANT types can store data structures like JSON or XML without predefined schema constraints, offering high flexibility for data ingestion and manipulation. - Integration: Provides better integration with systems that use semi-structured data, allowing for more direct data exchanges and queries

####1. Requirement
Read data from students_offline.csv file and load into offline_students_raw table.

In [0]:
offline_students_schema = "id string, first_name string, last_name string, address string, skills string, contacts string"

offline_students_raw_df = (
    spark.read.format("csv")
        .option("header", "true")
        .option("quote", "\"")
        .option("escape", "\"")
        .schema(offline_students_schema)
        .load("/Volumes/dev/spark_db/datasets/spark_programming/data/students_offline.csv")
)

#offline_students_raw_df.display()
offline_students_raw_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dev.spark_db.offline_students_raw")

####2. Requirement
Prepare an offline_var_students table which is ready for analysis


In [0]:
from pyspark.sql.functions import parse_json

#We used from_json beforeon 06. working with complex data. But now we can use parse_json. We don't need to require schema anymore like we did in from_json. parse_json will automatically infer schema(address:variant skills:variant) and type will be variant

offline_students_df = (
    offline_students_raw_df.withColumns({
        "address": parse_json("address"),
        "skills": parse_json("skills"),
        "contacts": parse_json("contacts")
    })
)

offline_students_df.display()
#offline_students_df.write.mode("overwrite").saveAsTable("dev.spark_db.offline_var_students")

####3. Requirement
Perform the following analysis
1. What is country wise student count.
2. Find all students with more than 1 years of Spark knowledge
3. Find all students who didn't provide phone or whatsapp

2.1 What is country wise student count.

In [0]:
%sql
-- Variant object element names are case sensitive
-- Variant cannot be used in order by and group by so we have to first cast it to required Data Type.

select cast(address:Country as string), count(*) as count
from dev.spark_db.offline_var_students
group by cast(address:Country as string)

2.2 Find all students with more than 1 years of Spark knowledge

In [0]:
%sql

with offline_students_skills(
  select id, first_name, last_name, cast(value:Skill as string), cast(value:YearsOfExperience as int)
  from dev.spark_db.offline_var_students, lateral variant_explode_outer(skills)
)
select *
from offline_students_skills
where skill like "%Spark%" and yearsofexperience>1

2.3 Find all students who didn't provide phone or whatsapp

In [0]:
%sql

select id, first_name, last_name, contacts:email
from dev.spark_db.offline_var_students
where contacts:phone is null and contacts:whatsapp is null